# Setup

In [1]:
!pip install torch
!pip install tensorflow

In [2]:
import os

# 1. Configurar Keras con Backend de TensorFlow
os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import keras
import keras_hub

# Activar crecimiento de memoria para evitar errores OOM
'''
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
'''

"\ngpus = tf.config.list_physical_devices('GPU')\nif gpus:\n    for gpu in gpus:\n        tf.config.experimental.set_memory_growth(gpu, True)\n"

# Dataset

Es muy importante que el corpus de ajuste fino este en el formato con el que se entreno el modelo base. A continuacion hay dos transformaciones para el caso en que el dataset este en formato usuario-asistente o en formato chatml.

In [ ]:
# VIEJO --- Para corpus en formato user assistant role:
import json
import tensorflow as tf

def cargar_y_formatear_chatml(jsonl_path):
    textos_formateados = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            string_conversacion = ""

            # Unimos los mensajes usando los tokens especiales de Qwen (ChatML)
            for msg in data["messages"]:
                role = msg["role"]
                content = msg["content"]
                string_conversacion += f"<|im_start|>{role}\n{content}<|im_end|>\n"

            textos_formateados.append(string_conversacion)
    print(f'Casos procesados: {len(textos_formateados)}')
    # Creamos el dataset de TensorFlow que KerasHub entiende de forma nativa
    dataset = tf.data.Dataset.from_tensor_slices(textos_formateados)

    # CRUCIAL: Duplicamos el texto para que actúe como entrada (x) y objetivo (y)
    # KerasHub se encargará de tokenizar y desfasar un token de forma interna
    dataset = dataset.map(lambda x: (x, x))

    # por lo que solo necesitamos mezclar y empaquetar en batches.
    # Hay que mezclar entre epocas de entrenamiento
    return dataset.shuffle(buffer_size=1000).batch(1)

# Reemplaza esto antes de tu código de entrenamiento:
train_dataset = cargar_y_formatear_chatml("corpus_chatbot_guia_metodos_numericos.jsonl")
print(f"Se cargaron {len(train_dataset)} casos para entrenar")

FileNotFoundError: [Errno 2] No such file or directory: 'corpus_chatbot_guia_metodos_numericos.jsonl'

In [3]:
#ACTUAL --- Para corpus en formato chatml
import tensorflow as tf
import pickle
import re

#Importar lista con casos:
with open("finetune_data.pickle", "rb") as infile:
    textos_originales = pickle.load(infile)

print(f"Casos cargados desde pickle: {len(textos_originales)}")

def limpiar_formato_chatml(texto_chatml):
    """Extrae el contenido y elimina los tokens especiales conflictivos."""
    match_user = re.search(r"<\|im_start\|>user\n(.*?)<\|im_end\|>", texto_chatml, re.DOTALL)
    match_assistant = re.search(r"<\|im_start\|>assistant\n(.*?)<\|im_end\|>", texto_chatml, re.DOTALL)

    if match_user and match_assistant:
        user_content = match_user.group(1).strip()
        assistant_content = match_assistant.group(1).strip()
        return f"User: {user_content}\nAssistant: {assistant_content}"
    return texto_chatml

# Aplicar la limpieza de formato chatml
textos_limpios = [limpiar_formato_chatml(t) for t in textos_originales]

print("\n--- Ejemplo de texto limpio ---")
print(textos_limpios[0])
print("-------------------------------\n")

# Crear el pipeline de datos en TensorFlow
BATCH_SIZE = 4
train_dataset = (
    tf.data.Dataset.from_tensor_slices(textos_limpios)
    .shuffle(buffer_size=1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Creamos el dataset de TensorFlow que KerasHub entiende de forma nativa
#dataset = tf.data.Dataset.from_tensor_slices(textos_formateados)

# CRUCIAL: Duplicamos el texto para que actúe como entrada (x) y objetivo (y)
# KerasHub se encargará de tokenizar y desfasar un token de forma interna
#dataset = dataset.map(lambda x: (x, x))

# por lo que solo necesitamos mezclar y empaquetar en batches.
# Hay que mezclar entre epocas de entrenamiento
#train_dataset = dataset.shuffle(buffer_size=1000).batch(4)

print(f"Se cargaron {len(train_dataset)} bloques para entrenar")

Casos cargados desde pickle: 142

--- Ejemplo de texto limpio ---
User: ¿Qué es exactamente un algoritmo y por qué la cátedra utiliza pseudocódigo en lugar de un lenguaje de programación específico?
Assistant: Un algoritmo es una forma ordenada de describir un procedimiento, y su expresión más elemental es el pseudocódigo, el cual constituye la expresión en palabras y ecuaciones de los pasos a seguir sin atarse a la sintaxis rígida de un lenguaje formal. Si observas la estructura del problema a resolver, ¿por qué crees que diseñar primero el pseudocódigo facilita luego la traducción a cualquier lenguaje como GNU Octave o C++?
-------------------------------

Se cargaron 36 bloques para entrenar


#Configuración del modelo

In [4]:
# 1. Clear any leftover backend memory from the previous crash
keras.backend.clear_session()

#https://keras.io/keras_hub/presets/
# Use float16 (T4 does not support bfloat16)
#keras.config.set_floatx("float32")
keras.mixed_precision.set_global_policy("mixed_float16")
model_preset = "qwen3_1.7b_en"
causal_lm = keras_hub.models.Qwen3CausalLM.from_preset(
    model_preset
)

100%|██████████| 653/653 [00:00<00:00, 1.48MB/s]


100%|██████████| 3.91k/3.91k [00:00<00:00, 8.70MB/s]


100%|██████████| 4.44M/4.44M [00:01<00:00, 2.44MB/s]


100%|██████████| 1.59M/1.59M [00:01<00:00, 1.10MB/s]


100%|██████████| 6.41G/6.41G [06:58<00:00, 16.4MB/s]


In [5]:
# Enable QLoRA with higher, more accurate settings
# Since sequence_length is capped, we have plenty of VRAM for int8 and rank=8.
#causal_lm.quantize("int8")
causal_lm.preprocessor.sequence_length = 256 #Limit context for OOM
causal_lm.backbone.enable_lora(rank=8)

In [6]:
# Compilar el modelo
causal_lm.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()]
)

#Entrenamiento

In [7]:
# Ajustar el modelo (Aumentar epochs según el tamaño real de tus datos)
causal_lm.fit(
    train_dataset,
    epochs=10, verbose=1
)

Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 212s 2s/step - loss: 1.7649 - sparse_categorical_accuracy: 0.4744
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - loss: 1.5393 - sparse_categorical_accuracy: 0.4854  
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - loss: 1.3682 - sparse_categorical_accuracy: 0.5167  
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - loss: 1.2349 - sparse_categorical_accuracy: 0.5405  
Epoch 5/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - loss: 1.1347 - sparse_categorical_accuracy: 0.5626  
Epoch 6/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - loss: 1.0594 - sparse_categorical_accuracy: 0.5811  
Epoch 7/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - loss: 0.9891 - sparse_categorical_accuracy: 0.5975  
Epoch 8/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - loss: 0.9242 - sparse_categorical_accuracy: 0.6176  
Epoch 9/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - loss: 0.8570 - sparse_categorical_accuracy: 0.6403  
Epoch 10/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - los

In [8]:
#TESTING

# Define Test Prompt using the training template structure
user_query = "¿Qué es exactamente un algoritmo y por qué la cátedra utiliza pseudocódigo en lugar de un lenguaje de programación específico?"

formatted_prompt = (
    f"User: {user_query}\nAssistant:"
)

# Generate Response
causal_lm.sampler = keras_hub.samplers.GreedySampler()
# Max length includes prompt + output tokens
output = causal_lm.generate(
    formatted_prompt,
    max_length=256
)

print("--- GENERATED RESPONSE ---")
print(output)

generated token ids =  Tensor("strided_slice_28:0", shape=(256,), dtype=int32)
--- GENERATED RESPONSE ---
User: ¿Qué es exactamente un algoritmo y por qué la cátedra utiliza pseudocódigo en lugar de un lenguaje de programación específico?
Assistant: Un algoritmo es una secuencia de pasos lógicos que resuelve un problema específico. El pseudocódigo es una representación abstracta de esta secuencia, usando símbolos y estructuras de lenguaje humano para describir el flujo de información. La cátedra lo utiliza para facilitar la comprensión conceptual antes de abordar la codificación con lenguajes específicos. Analiza esta definición y explica por qué el pseudocódigo es una herramienta ideal para la enseñanza de algoritmos.


In [9]:
preguntas=[]
preguntas.append('Escribí mi algoritmo empezando directamente con las operaciones matemáticas, pero el profesor me dijo que me faltan secciones obligatorias. ¿Cuáles son?')
preguntas.append('No me queda claro qué representa físicamente una variable en la computadora cuando la defino en mi programa.')
preguntas.append('Tengo la expresión combinada res <- x + y * z ^ 2. ¿En qué orden exacto resolverá los operadores el compilador?')

respuestas_dataset=[]
respuestas_dataset.append('Todo algoritmo debe contar obligatoriamente con cuatro elementos estructurados: declaración de variables, ingreso de datos o asignaciones primarias, el proceso propiamente dicho y la entrega de resultados. Si omites declarar las variables al inicio del pseudocódigo, ¿cómo sabría la computadora qué recipientes y direcciones de memoria debe reservar antes de procesar la información?')
respuestas_dataset.append('Una variable se define como una especie de recipiente con una dirección alojada en la memoria de la computadora para su identificación interna. Para que nosotros los programadores la podamos identificar, le asignamos un nombre representativo del contenido que guarda. Pensando en la memoria como un conjunto de casilleros etiquetados, ¿qué ocurriría si intentaras buscar o guardar un dato en un casillero que no tiene dirección ni etiqueta definida?')
respuestas_dataset.append('Revisando la tabla de prioridades: 1º se resuelve la potencia z ^ 2 (Nivel 1), 2º el producto y * (z^2) (Nivel 2), y 3º la suma x + ... (Nivel 3). Todos los resultados parciales de estas operaciones son numéricos. Si tu objetivo era multiplicar y * z y elevar todo ese producto al cuadrado antes de sumar x, ¿cómo deberías agrupar los términos?')

causal_lm.sampler = keras_hub.samplers.GreedySampler()
i=0
for pregunta in preguntas:
    formatted_prompt = (
    f"User: {pregunta}\nAssistant:"
    )
    output = causal_lm.generate(
      formatted_prompt,
      max_length=512
     )
    print(f"########## CASO {i} ##########")
    print("--- GENERATED RESPONSE ---")
    print(output)
    print("--- TRAINDATA RESPONSE ---")
    print(respuestas_dataset[i])
    i+=1


generated token ids =  Tensor("strided_slice_28:0", shape=(512,), dtype=int32)
########## CASO 0 ##########
--- GENERATED RESPONSE ---
User: Escribí mi algoritmo empezando directamente con las operaciones matemáticas, pero el profesor me dijo que me faltan secciones obligatorias. ¿Cuáles son?
Assistant: Todas las operaciones matemáticas deben estar acompañadas de una asignación de variable para almacenar el resultado. Si no asignas una variable, el ordenador no sabe qué valor guarda tu cálculo. Revisa la teoría del capítulo sobre la importancia de las variables y cómo se estructuran las instrucciones en el algoritmo. ¿Qué estructura de código debes añadir al final de tu algoritmo para guardar el resultado de la suma?
--- TRAINDATA RESPONSE ---
Todo algoritmo debe contar obligatoriamente con cuatro elementos estructurados: declaración de variables, ingreso de datos o asignaciones primarias, el proceso propiamente dicho y la entrega de resultados. Si omites declarar las variables al inic

In [10]:
print("--- COMPILACIÓN DEL MODELO ---")
causal_lm.summary()

--- COMPILACIÓN DEL MODELO ---


Preprocessor: "qwen3_causal_lm_preprocessor_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ qwen3_tokenizer (Qwen3Tokenizer)                              │                      Vocab size: 151,669 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "qwen3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ qwen3_backbone                │ (None, None, 2048)        │   1,731,642,368 │ padding_mask[0][0],        │
│ (Qwen3Backbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 151936)      │     311,164,928 │ qwen3_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,753,777,158 (6.53 GB)

 Trainable params: 11,067,392 (42.22 MB)

 Non-trainable params: 1,720,574,976 (6.41 GB)

 Optimizer params: 22,134,790 (84.44 MB)

In [13]:
#Guardar adapter
import numpy as np

# 1. Filtrar únicamente los pesos de LoRA
lora_weights = {
    weight.path: weight.numpy()
    for weight in causal_lm.weights
    if "lora" in weight.name.lower() or "lora" in weight.path.lower()
}

# 2. Guardar en un archivo .npz
adapter_path = "qwen3_lora_adapter.npz"
np.savez(adapter_path, **lora_weights)

print(f"Adapter guardado exitosamente en {adapter_path} con {len(lora_weights)} tensores.")

Adapter guardado exitosamente en qwen3_lora_adapter.npz con 112 tensores.


In [ ]:
import numpy as np

# 1. Reconstruir modelo base y activar LoRA
causal_lm = keras_hub.models.Qwen3CausalLM.from_preset("qwen3_1.7b_en")
causal_lm.backbone.enable_lora(rank=8)

# 2. Cargar el archivo .npz
loaded_weights = np.load("qwen3_lora_adapter.npz")

# 3. Asignar los valores a las variables de LoRA del modelo
weight_dict = {w.path: w for w in causal_lm.weights}
for path, array in loaded_weights.items():
    if path in weight_dict:
        weight_dict[path].assign(array)

print("Adapter LoRA recargado exitosamente.")

In [ ]:
# ----------------------------------------------------------------------
# Guardar los Pesos Entrenados de LoRA
# ----------------------------------------------------------------------
output_filename = "lora_weights_cathedra.weights.h5"
causal_lm.save_weights(output_filename)
print(f"\n¡Entrenamiento completado! Pesos guardados en: {output_filename}")

Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x7d4f9b1567a0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):
KeyboardInterrupt: 



¡Entrenamiento completado! Pesos guardados en: lora_weights_cathedra.weights.h5


In [ ]:
#PARA RECUPERAR EL MODELO
import keras_hub

# 1. Cargar el modelo base
model = keras_hub.models.CausalLM.from_preset("qwen3_1.7b_en")
model.preprocessor.sequence_length = 256

# 2. Habilitar LoRA con el mismo rank utilizado en el entrenamiento
model.backbone.enable_lora(rank=8)

# 3. Cargar tus pesos guardados
model.load_weights("lora_weights_cathedra.weights.h5")

In [ ]:
# Guardar el modelo completo en formato .keras
causal_lm.save("modelo_completo_finetuned.keras")

In [ ]:
import keras

# Cargar directamente el modelo con todas sus capas y preprocesador
model = keras.models.load_model("modelo_completo_finetuned.keras")

# Testing

In [ ]:
def test_model_tutor(pregunta_estudiante):
    # 1. Construir el prompt EXACTO con el formato ChatML que usamos en el entrenamiento
    prompt = (
        f"<|im_start|>system\n"
        f"Eres un chatbot del curso de Métodos Numéricos en Ingenieria. Respondes preguntas de estudiantes sobre la guia de estudio.<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{pregunta_estudiante}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    # 2. Generar la respuesta del modelo
    # Subimos un poco max_length por si el tutor se extiende en su explicación socrática
    response = causal_lm.generate(prompt, max_length=512)

    # 3. Limpieza: Extraer solo lo que respondió el asistente (para no reimprimir todo el prompt)
    try:
        assistant_reply = response.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
    except Exception:
        assistant_reply = response # Backup por si los tokens de parada fallan

    return assistant_reply

# --- Casos de Prueba Reales para tu Tutor Socrático ---
test_cases = [
    # Caso 1: La fórmula es r = (a + b) / 2, donde 'a' y 'b' son los extremos del intervalo.
    "Profe, ¿Cuál es la fórmula de recurrencia del método de Bisección?",

    # Caso 2: Son conjuntos de órdenes que constituyen todo un proceso identificadas con un nombre particular. Pueden recibir o entregar argumentos de entrada y salida.
    "¿Para qué sirven los Subprogramas?"
]

# --- Ejecución de las pruebas ---
print("=== EVALUANDO RESPUESTAS DEL TUTOR SOCRÁTICO ===")
for estudiante_input in test_cases:
    reply = test_model_tutor(estudiante_input)
    print(f"\n[ESTUDIANTE]: {estudiante_input}")
    print(f"[TUTOR IA]:   {reply}")
    print("-" * 80)

=== EVALUANDO RESPUESTAS DEL TUTOR SOCRÁTICO ===
generated token ids =  Tensor("strided_slice_28:0", shape=(512,), dtype=int32)

[ESTUDIANTE]: Profe, ¿Cuál es la fórmula de recurrencia del método de Bisección?
[TUTOR IA]:   </think>Esystem
Eres un chatbot del curso de Métodos Numéricos en Ingenieria. Respondes preguntas de estudiantes sobre la guia de estudio.<|box_end|>J
</think>Euser
Profe, ¿Cuál es la fórmula de recurrencia del método de Bisección?<|box_end|>J
</think>Eassistant
Se define la función f(x), se toman dos valores, x0 y x1, tales que f(x0) y f(x1) se cambian de signo y se define una variable, x2 = (x0 + x1)/2. La función f(x2) se evalúa para verificar si el cambio de signo ocurre en x2. Si ocurre, se define x0 = x1 y x1 = x2, o viceversa, según sea el signo de f(x2).<|box_end|>J
--------------------------------------------------------------------------------

[ESTUDIANTE]: ¿Para qué sirven los Subprogramas?
[TUTOR IA]:   </think>Esystem
Eres un chatbot del curso de Métod